In [ ]:
# EDA tambahan: beban jangkauan rata-rata per kabupaten/kota.
coverage_load = summary.sort_values('area_per_district_km2')
ax = coverage_load['area_per_district_km2'].plot(kind='barh', figsize=(10, 4), color='#2a6f97', title='Beban jangkauan rata-rata per ADM2')
ax.set(xlabel='Rata-rata luas per kabupaten/kota (km²)', ylabel='Province')
plt.show()
widest_territory = coverage_load.index[-1]
print(f'Business insight: {widest_territory} memiliki beban jangkauan rata-rata per ADM2 terbesar. Gunakan ini untuk menilai kebutuhan hub-and-spoke, pos logistik, atau dukungan bergerak saat kesiapsiagaan karhutla naik—bukan sebagai peringkat risiko kebakaran.')

# geoBoundaries: operational coverage and coordination complexity

**Business question.** How should regional coverage be organised across provinces and districts? This notebook uses geoBoundaries ADM1/ADM2 geometry only—no fire detections, population, or model results.

**Decision use.** Balance field-team territories, district coordination load, and reporting cadence.

In [ ]:
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

ROOT = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / 'data').exists())
adm1 = gpd.read_file(ROOT / 'data/raw/boundaries/idn_adm1.geojson')
adm2 = gpd.read_file(ROOT / 'data/raw/boundaries/idn_adm2.geojson')
province_col = next(c for c in ['shapeName', 'NAME_1', 'name'] if c in adm1.columns)
kalimantan = adm1[adm1[province_col].astype(str).str.contains('Kalimantan', case=False, na=False)].copy()
# Associate ADM2 spatially: shapeGroup is a country code, not a province name.
province_key = 'province_name'
province_shapes = kalimantan[[province_col, 'geometry']].rename(columns={province_col: province_key})
districts = gpd.sjoin(adm2.to_crs(kalimantan.crs), province_shapes, how='inner', predicate='intersects')
if 'shapeID' in districts.columns:
    districts = districts.drop_duplicates('shapeID')
equal_area = kalimantan.to_crs('EPSG:6933')
summary = pd.DataFrame({
    'province_area_km2': equal_area.set_index(province_col).area / 1e6,
    'district_count': districts.groupby(province_key).size(),
}).fillna(0).sort_values('province_area_km2', ascending=False)
summary['area_per_district_km2'] = summary['province_area_km2'] / summary['district_count'].replace(0, pd.NA)
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
summary[['province_area_km2', 'district_count']].plot(kind='bar', secondary_y='district_count', ax=axes[0], color=['#2a6f97', '#f4a261'])
axes[0].set(title='Geographic coverage versus coordination load', xlabel='Province', ylabel='Area (km²)')
kalimantan.to_crs('EPSG:4326').plot(ax=axes[1], color='#d9eaf7', edgecolor='white')
districts.to_crs('EPSG:4326').boundary.plot(ax=axes[1], color='#4a4a4a', linewidth=.25)
axes[1].set_title('Kalimantan province and district operating map')
axes[1].set_axis_off()
plt.show()

largest_area = summary['province_area_km2'].idxmax()
most_districts = summary['district_count'].idxmax()
print(f'Business insight: {largest_area} has the largest physical coverage requirement, while {most_districts} has the highest district coordination load.')
print('Recommended action: size patrol/logistics territories from area, but size stakeholder-management capacity from district count; these are different operating constraints.')

## Decision notes

- Large provinces need logistics reach even when they have few administrative counterparts.
- Provinces with many districts need a stronger coordination and reporting cadence.
- This is an operating-coverage analysis, not a risk ranking.